### Contents  
1 Import libraries and datasets  
2 Inspect schools data and clean  
3 Inspect population data and clean  
4 Inspect population density data and clean  
5 Export datasets

### 1 Import libraries and datasets

In [69]:
# import libraries
import pandas as pd
import numpy as np
import os

In [70]:
# create path
path = r"C:\Users\cathe\OneDrive\Data Analysis\2 6 Schools in England and Wales\02 Data"

In [71]:
# import dataset 1: schools information and inspection results
df1 = pd.read_csv(os.path.join(path, "Original Data", "Schools_2025.csv"), encoding='utf-8', encoding_errors='ignore', index_col=False)

In [72]:
# import dataset 2: population census data
df2 = pd.read_csv(os.path.join(path, "Original Data", "Population.csv"), index_col=False)

In [73]:
# import dataset 3: population densities
df3 = pd.read_csv(os.path.join(path, "Original Data", "Densities.csv"), index_col=False)

### 2 Inspect schools data and clean

In [75]:
df1.shape

(21990, 38)

In [104]:
# Investigate quantitative columns
df1.describe()

,URN,The income deprivation affecting children index (IDACI) quintile,Total number of pupils,Quality of education,Behaviour and attitudes,Personal development,Effectiveness of leadership and management,Previous graded inspection overall effectiveness,Previous quality of education,Previous behaviour and attitudes,Previous personal development,Previous effectiveness of leadership and management
count,21901.000000,21901.000000,21901.000000,21802.000000,21802.000000,21802.000000,21802.000000,21196.000000,21196.000000,21196.000000,21196.000000,21196.000000
mean,129521.502397,3.029679,386.787864,5.238418,5.132924,5.097560,1.919824,2.233393,8.642291,8.612002,8.602236,2.138658
std,16048.386817,1.409568,365.418831,3.481080,3.576773,3.606826,0.521623,0.814599,1.423489,1.547809,1.584745,0.796118
min,100000.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,115137.000000,2.000000,166.000000,2.000000,2.000000,2.000000,2.000000,2.000000,9.000000,9.000000,9.000000,2.000000
50%,136469.000000,3.000000,247.000000,3.000000,3.000000,2.000000,2.000000,2.000000,9.000000,9.000000,9.000000,2.000000
75%,143452.000000,4.000000,446.000000,9.000000,9.000000,9.000000,2.000000,3.000000,9.000000,9.000000,9.000000,3.000000
max,151631.000000,5.000000,3077.000000,9.000000,9.000000,9.000000,4.000000,4.000000,9.000000,9.000000,9.000000,4.000000


Comments:
- URN: all six digit numbers with 21990 distinct values
- IDACI: quartiles as expected; 89 missing entries
- Total number of pupils: min 0 is an error, needs investigating and removing; max is large and should be checked; 47 missing values
- Other URN columns: looks like URNs have changed, but only using them to identify distinct schools, so will delete these columns
- Overall, quality, behaviour, personal, effectiveness: see if can find rationale for numbers.  Why is max 9 for some and 4 for others?
- Overall effectiveness: check that column for latest inspection also in dataframe as has not appeared in above table.  Data type issue?

In [82]:
# Remove all rows containing missing IDACI quintile
df1.dropna(subset=["The income deprivation affecting children index (IDACI) quintile"], inplace=True)

Note: All missing values for Total number of pupils have also been removed.  Minimum value for Total number of pupils is now 1.

In [84]:
df1.shape

(21901, 38)

In [112]:
# Investigate schools with 1 pupil
no_pupils = df1[df1["Total number of pupils"] == 1]
print(no_pupils[["Total number of pupils", "Type of education"]])

       Total number of pupils                        Type of education
1671                      1.0                      Pupil Referral Unit
9617                      1.0                      Pupil Referral Unit
9974                      1.0                      Pupil Referral Unit
10238                     1.0                      Pupil Referral Unit
10244                     1.0                      Pupil Referral Unit
10346                     1.0                      Pupil Referral Unit
10369                     1.0                      Pupil Referral Unit
10414                     1.0                      Pupil Referral Unit
10557                     1.0                      Pupil Referral Unit
11936                     1.0                      Pupil Referral Unit
12392                     1.0      Free School - Alternative Provision
14644                     1.0      Free School - Alternative Provision
15707                     1.0  Academy Alternative Provision Converter
18917 

PRUs are often very small and alternative provision similar, so I will leave these records as they are.

In [95]:
# Investigate schools with over 2000 pupils
many_pupils = df1[df1["Total number of pupils"] > 2000]
print(many_pupils["Total number of pupils"])

95       2080.0
595      2323.0
597      2160.0
598      2900.0
839      2015.0
1237     2207.0
3135     2518.0
4277     2196.0
5341     2507.0
6123     2002.0
10502    2367.0
10597    2003.0
10674    2188.0
10727    2356.0
10845    2527.0
10870    2424.0
10933    2467.0
10957    2490.0
10992    2097.0
11058    2094.0
11090    2187.0
11144    2106.0
11146    2880.0
11211    2126.0
11220    2121.0
11264    3077.0
11275    2122.0
11321    2134.0
11401    2275.0
11420    2024.0
11442    2273.0
11476    2158.0
11519    2068.0
11537    2080.0
11780    2040.0
11796    2052.0
11944    2008.0
12022    2060.0
12151    2013.0
12162    2645.0
12319    2060.0
12558    2014.0
12695    2276.0
13079    2084.0
13440    2178.0
13501    2056.0
14025    2094.0
14114    2779.0
14190    2628.0
14249    2050.0
15057    2593.0
16938    2471.0
18928    2037.0
19592    2223.0
19877    2116.0
20821    2201.0
Name: Total number of pupils, dtype: float64


3077 is not a long way from other school sizes, so assume it is correct

In [51]:
df1.dtypes

URN                                                                               int64
Ofsted phase                                                                     object
Type of education                                                                object
School open date                                                                 object
Admissions policy                                                                object
Sixth form                                                                       object
Region                                                                           object
Local authority                                                                  object
Postcode                                                                         object
The income deprivation affecting children index (IDACI) quintile                float64
Total number of pupils                                                          float64
Inspection number of latest grad

In [57]:
# Investigate why Overall effectiveness column is dtype object, when previous overall effectiveness is float64
df1["Overall effectiveness"].value_counts()

Overall effectiveness
2             16053
1              2590
3              1552
Not judged     1378
4               270
Name: count, dtype: int64

I will leave "Not judged" in the dataset, since it accounts for over 5% of the dataset and might introduce bias if I remove these schools

In [100]:
# Drop unwanted columns which reference URN
df1 = df1.drop(["Does the latest graded inspection relate to the URN of the current school?", "URN at time of latest graded inspection", "Does the previous graded inspection relate to the URN of the current school?", "URN at time of previous graded inspection"], axis=1)

In [102]:
df1.shape

(21901, 34)

In [119]:
# Consistency checks for qualitative columns (looping through all columns for simplicity)
# Retrieve all column names as a list
columns = df1.columns

# Loop through each column and print value counts
for column in columns:
    print(f"Value counts for {column}:")
    print(df1[column].value_counts(dropna=False))
    print("\n")

Value counts for URN:
URN
100000    1
141101    1
141111    1
141110    1
141109    1
         ..
119369    1
119368    1
119367    1
119366    1
151631    1
Name: count, Length: 21901, dtype: int64


Value counts for Ofsted phase:
Ofsted phase
Primary      16719
Secondary     3412
Special       1090
Nursery        379
PRU            301
Name: count, dtype: int64


Value counts for Type of education:
Type of education
Academy Converter                            7343
Community School                             5277
Academy Sponsor Led                          2654
Voluntary Aided School                       2230
Voluntary Controlled School                  1441
Foundation School                             613
Free School                                   507
Community Special School                      446
LA Nursery School                             379
Academy Special Converter                     317
Pupil Referral Unit                           149
Free School Special         

Comments:
- Consistent naming and no unusual results for: Ofsted phase, School open date, Admissions policy, Sixth form, Local authority. 
- Type of education, Event type grouping, Category of concern, Previous category of concern - to be deleted as too granular/technical for my project
- Region: where is Wales in the list?

In [122]:
# Delete unneeded columns mentioned above
df1 = df1.drop(["Type of education", "Event type grouping", "Category of concern", "Previous category of concern"], axis=1)

In [124]:
df1.shape

(21901, 30)

### 3 Inspect population data and clean

In [127]:
df2.shape

(357, 95)

In [133]:
df2.head()

,Code,Name,Geography,All ages,0,1,2,3,4,5,...,81,82,83,84,85,86,87,88,89,90+
0,K04000001,ENGLAND AND WALES,Country,"60,854,727","600,801","641,879","638,385","661,083","672,685","681,900",...,"335,360","288,359","287,413","271,872","250,133","222,324","195,657","170,123","142,674","551,758"
1,E92000001,ENGLAND,Country,"57,690,323","573,100","611,983","608,924","629,972","640,658","648,548",...,"314,801","270,597","270,666","256,427","235,873","209,664","184,536","160,616","134,632","521,291"
2,E12000001,NORTH EAST,Region,"2,711,380","24,711","26,256","26,370","26,959","28,242","28,768",...,"15,345","13,718","13,665","12,776","11,874","10,641","8,949","8,003","6,504","23,914"
3,E06000047,County Durham,Unitary Authority,"532,182","4,410","4,736","4,805","4,964","5,084","5,199",...,"3,180","2,922","2,790","2,542","2,361","2,164","1,825","1,596","1,284","4,497"
4,E06000005,Darlington,Unitary Authority,"110,562","1,010","1,115","1,102","1,157","1,168","1,210",...,633,592,613,554,503,448,377,354,277,"1,083"


In [129]:
df2.describe()

,Code,Name,Geography,All ages,0,1,2,3,4,5,...,81,82,83,84,85,86,87,88,89,90+
count,357,357,357,357,357,357,357,357,357,357,...,357,357,357,357,357,357,357,357,357,357
unique,357,357,8,357,341,335,335,345,342,342,...,327,317,318,311,310,307,304,291,282,333
top,K04000001,ENGLAND AND WALES,Non-metropolitan District,"60,854,727","1,051","1,282","1,283","1,040","1,266","1,332",...,860,643,481,651,501,634,470,307,345,"1,153"
freq,1,1,164,1,3,2,3,3,2,2,...,3,4,4,4,3,4,3,4,4,3


Comments:
- all counts are 357 so no missing data

### 4 Inspect population density data and clean

In [137]:
df3.shape

(331, 3)

In [139]:
df3.head()

,Lower Tier Local Authorities Code,Lower Tier Local Authorities,Observation
0,E06000001,Hartlepool,985.5
1,E06000002,Middlesbrough,2671.2
2,E06000003,Redcar and Cleveland,557.1
3,E06000004,Stockton-on-Tees,959.3
4,E06000005,Darlington,545.9


In [141]:
df3.describe()

,Observation
count,331.000000
mean,1713.796375
std,2482.096656
min,25.500000
25%,228.700000
50%,615.100000
75%,2298.400000
max,15702.900000


Comments:
- minimum is reasonable
- maximum is 157 people per 100m by 100m square, which seems reasonable

In [143]:
# Rename observation column
df3.rename(columns = {"Observation" : "Population Density"}, inplace = True)

In [148]:
df3.sample(5)

,Lower Tier Local Authorities Code,Lower Tier Local Authorities,Population Density
276,E09000001,City of London,2975.0
39,E06000042,Milton Keynes,930.1
26,E06000027,Torbay,2215.6
242,E08000003,Manchester,4772.7
133,E07000112,Folkestone and Hythe,307.5


### 5 Export datasets

In [152]:
df1.to_csv(os.path.join(path, "Prepared Data", "schools_cleaned.csv"))

In [154]:
df2.to_csv(os.path.join(path, "Prepared Data", "population_cleaned.csv"))

In [ ]:
df3.to_csv(os.path.join(path, "Prepared Data", "densitites_cleaned.csv"))